# 4.7 Fuzzy c-means

Fuzzy c-means uses soft memberships instead of forcing every point to make an immediate hard choice.

Bayes-style normalization and weighted averages become the mechanism: normalize scores into responsibilities, then update representatives with those weights. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler

RNG = np.random.default_rng(7)


def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs


def scaled(X):
    return StandardScaler().fit_transform(X)


def plot_xy(X):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=0).fit_transform(X)


def safe_silhouette(X, labels):
    if len(np.unique(labels)) < 2:
        return float("nan")
    if len(np.unique(labels)) >= len(labels):
        return float("nan")
    return float(silhouette_score(X, labels))

def fuzzy_c_means(X, c, m=2.0, max_iter=80, tol=1e-4, seed=0):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    U = rng.random((n, c))
    U = U / U.sum(axis=1, keepdims=True)

    for step in range(max_iter):
        U_old = U.copy()
        weights = U ** m
        centers = weights.T @ X
        centers = centers / weights.sum(axis=0)[:, None]
        distances = pairwise_distances(X, centers)
        distances = np.maximum(distances, 1e-12)
        power = 2.0 / (m - 1.0)
        ratio = distances[:, :, None] / distances[:, None, :]
        U = 1.0 / np.sum(ratio ** power, axis=2)
        shift = np.linalg.norm(U - U_old)
        if shift < tol:
            break

    return U, centers


## The concept, built once on D1

Fuzzy c-means optimizes soft cluster representatives. The lesson's arithmetic is

$$r_{ik}=\frac{q_{ik}}{\sum_j q_{ij}},\qquad \mu_k=\frac{\sum_i r_{ik}x_i}{\sum_i r_{ik}}$$

The first cell audits the exact normalization and weighted mean from the lesson before we use the reusable method.

In [ ]:

scores = np.array([0.30, 0.70])
responsibilities = scores / scores.sum()
assert np.allclose(responsibilities, [0.30, 0.70])

x = np.array([0.0, 1.0, 4.0])
weights = np.array([0.8, 0.6, 0.1])
weighted_mean = np.sum(weights * x) / np.sum(weights)
assert round(float(weighted_mean), 3) == 0.667

print("responsibilities", responsibilities)
print("weighted mean", round(float(weighted_mean), 3))


Now the same weighted update is wrapped into `fuzzy_c_means`. The only difference on real rungs is that memberships are repeatedly recomputed from distances and then reused as weights for the next centers.

In [ ]:

X_d1 = scaled(cluster_ladder()[0][1])
U_d1, centers_d1 = fuzzy_c_means(X_d1, c=3, m=2.0, seed=0)
labels_d1 = np.argmax(U_d1, axis=1)

print("D1 memberships")
print(np.round(U_d1, 3))
print("D1 centers")
print(np.round(centers_d1, 3))
print("D1 labels", labels_d1.tolist())


## The dataset ladder

We use the shared F2 clustering ladder exactly once in the setup cell: hand points, clean blobs, anisotropic overlap, real Iris, and real digits 0–3. The hidden labels are kept only for audit metrics such as ARI; the clustering methods never train on them.

In [ ]:

rungs = cluster_ladder()

for idx, (name, X, y_true, k) in enumerate(rungs, start=1):
    print(idx, name, "shape=", X.shape, "k=", k, "labels=", np.unique(y_true).tolist())
    print("sample", np.round(X[:3, :min(4, X.shape[1])], 3))


## Run the same method across D1–D5

Each rung is scaled, clustered with the same fuzzy c-means implementation, hardened by maximum membership only for scoring, and summarized with silhouette plus ARI for audit.

In [ ]:

results = []
artifacts = []

for idx, (name, X, y_true, k) in enumerate(cluster_ladder(), start=1):
    Xs = scaled(X)
    U, centers = fuzzy_c_means(Xs, c=k, m=2.0, seed=idx)
    labels = np.argmax(U, axis=1)
    score = safe_silhouette(Xs, labels)
    ari = adjusted_rand_score(y_true, labels)
    ambiguity = float(np.mean(1.0 - np.max(U, axis=1)))
    results.append({"rung": idx, "name": name, "score": score, "ari": ari, "ambiguity": ambiguity})
    artifacts.append((Xs, labels, U, centers))

for row in results:
    print(row["rung"], row["name"], "silhouette=", round(row["score"], 3), "ARI=", round(row["ari"], 3), "ambiguity=", round(row["ambiguity"], 3))


## Results visualization

The closing figure has two parts: assignment panels with opacity tied to fuzzy confidence, then the silhouette curve from D1 to D5.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(18, 3))

for ax, row, artifact in zip(axes, results, artifacts):
    Xs, labels, U, centers = artifact
    coords = plot_xy(Xs)
    alpha = 0.25 + 0.75 * np.max(U, axis=1)
    ax.scatter(coords[:, 0], coords[:, 1], c=labels, cmap="tab10", alpha=alpha, s=18)
    ax.set_title(f"D{row['rung']} sil={row['score']:.2f}")
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()

plt.figure(figsize=(6, 3))
plt.plot([r["rung"] for r in results], [r["score"] for r in results], marker="o")
plt.xticks([1, 2, 3, 4, 5])
plt.xlabel("ladder rung")
plt.ylabel("silhouette")
plt.title("Fuzzy c-means silhouette vs complexity")
plt.show()


## Pitfall on D5: fuzzifier and scale can hide ambiguous digits

A very large fuzzifier spreads memberships until many digits look partly owned by every cluster. The fix is to scale the D5 features and sweep a small set of fuzzifier values instead of accepting one over-soft run.

In [ ]:

name, X5, y5, k5 = cluster_ladder()[-1]

U_raw, centers_raw = fuzzy_c_means(X5, c=k5, m=5.0, seed=11)
labels_raw = np.argmax(U_raw, axis=1)
raw_ambiguity = float(np.mean(1.0 - np.max(U_raw, axis=1)))
raw_silhouette = safe_silhouette(X5, labels_raw)

Xs5 = scaled(X5)
best = None

for m_value in [1.4, 1.7, 2.0, 2.5]:
    U_fix, centers_fix = fuzzy_c_means(Xs5, c=k5, m=m_value, seed=11)
    labels_fix = np.argmax(U_fix, axis=1)
    score_fix = safe_silhouette(Xs5, labels_fix)
    ambiguity_fix = float(np.mean(1.0 - np.max(U_fix, axis=1)))
    candidate = (score_fix, ambiguity_fix, m_value, labels_fix)
    if best is None or candidate[0] > best[0]:
        best = candidate

print("over-soft raw", round(raw_silhouette, 3), round(raw_ambiguity, 3))
print("best scaled sweep", round(best[0], 3), round(best[1], 3), "m=", best[2])


## Evaluate it + Practice

- Metric: track silhouette after hardening by max membership on every rung and compare against a no-skill baseline such as random labels with the same number of clusters.
- Sanity check: D1 and D2 should be visibly easier than D5; if not, inspect scaling and assignments before trusting the curve.
- Ablation: increase `m` until assignments become too soft, then compare with the scaled sweep
- Failure signals: memberships nearly uniform, silhouette flat, or ARI disagreeing with visible digit groups
- Baseline: random labels with the same k

Practice prompts:
1. Change one hyperparameter by a small amount and explain whether the D5 score moves for a meaningful reason.

2. Add a random-label baseline to the results table and compare it with the method.

3. Pick one D5 point, print its assignment evidence, and decide whether the method is confident or ambiguous.